### 1.comparaison entre images et leur corespondance (labels)

In [ ]:
#comparaison entre les images et les labels pour identifier les images sans annotation et les annotations sans image correspondante
from pathlib import Path

DATASET_DIR = Path("../trafic_data")

for split in ["train", "valid"]:
    images = list((DATASET_DIR / split / "images").glob("*"))
    labels = list((DATASET_DIR / split / "labels").glob("*.txt"))

    noms_images = {f.stem for f in images}   #f.stem récupère le nom du fichier sans extension.
    noms_labels = {f.stem for f in labels}

    print(f"\n{split.upper()} :")
    print("Images sans Labels :", len(noms_images - noms_labels))   #trouve les images sans Label correspondante
    print("label sans image :", len(noms_labels - noms_images))   #trouve labels sans image correspondante

### 2.Examiner le format YOLO

In [ ]:
#afficher le nombre d'annotations vides pour chaque split
for split in ["train", "valid"]:
    labels = list((DATASET_DIR / split / "labels").glob("*.txt"))
    vides = [f for f in labels if f.stat().st_size == 0]

    print(f"{split.upper()} : {len(vides)} annotations vides")
    
    
#found2 in Train 

In [ ]:
#Afficher la structure du fichier text d'annotation
label_path = next(
    fichier for fichier in (DATASET_DIR / "train" / "labels").glob("*.txt")
    if fichier.stat().st_size > 0    
)

print("Fichier :", label_path.name)
print(label_path.read_text())

In [ ]:
valeurs = label_path.read_text().splitlines()[0].split()

print("Value numbers :", len(valeurs))
print("Class ID :", valeurs[0])
print("X center  :", valeurs[1])
print("Y center :", valeurs[2])
print("width :", valeurs[3])
print("height :", valeurs[4])

In [ ]:
#vérifie la normalisation des coordonnées et des dimensions des bounding boxes au format YOLO.
x, y, largeur, hauteur = map(float, valeurs[1:]) # Convertir les valeurs en nombres

print("Centre X valide :", 0 <= x <= 1)
print("Centre Y valide :", 0 <= y <= 1)
print("Largeur valide :", 0 < largeur <= 1)
print("Hauteur valide :", 0 < hauteur <= 1)

In [ ]:
# Vérifier le fichier data_1.yaml, les noms des classes et les identifiants utilisés
import yaml

with open(DATASET_DIR / "data_1.yaml", "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

names = data_cfg["names"]
nc = data_cfg["nc"]
print(f"Chemins YAML : train={data_cfg.get('train')}, val={data_cfg.get('val')}, test={data_cfg.get('test')}")
print(f"Nombre de classes déclaré (nc) : {nc} | Nombre de noms : {len(names)}")
assert nc == len(names), "Incohérence entre nc et len(names)"
for idx, nom in enumerate(names):
    print(f"  {idx:2d}: {nom}")

for split in ["train", "valid", "test"]:
    lbl_dir = DATASET_DIR / split / "labels"
    if not lbl_dir.exists():
        continue
    class_ids = set()
    for label_file in lbl_dir.glob("*.txt"):
        for ligne in label_file.read_text(encoding="utf-8").splitlines():
            if ligne.strip():
                class_ids.add(int(ligne.split()[0]))
    manquantes = [names[i] for i in range(nc) if i not in class_ids]
    print(f"\n{split.upper()} -> {len(class_ids)}/{nc} classes utilisées")
    print("  Identifiants :", sorted(class_ids))
    print("  Classes absentes :", manquantes)

In [ ]:
# Vérifier les images corrompues sur tous les splits
from PIL import Image

for split in ["train", "valid", "test"]:
    dossier = DATASET_DIR / split / "images"
    if not dossier.exists():
        continue
    corrompues = []

    for fichier in dossier.iterdir():
        if fichier.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
            try:
                with Image.open(fichier) as img:
                    img.verify()
            except Exception:
                corrompues.append(fichier.name)

    print(split.upper(), ":", len(corrompues), "images corrompues")

In [ ]:
# Identifier les lignes invalides (ex: boîtes avec w=0 ou h=0)
for split in ["train", "valid", "test"]:
    labels_dir = DATASET_DIR / split / "labels"
    if not labels_dir.exists():
        continue
    invalides = []

    for fichier in labels_dir.glob("*.txt"):
        for n, ligne in enumerate(fichier.read_text(encoding="utf-8").splitlines(), 1):
            if not ligne.strip():
                continue

            v = ligne.split()
            if len(v) != 5:
                invalides.append(f"{fichier.name}, ligne {n}")
                continue

            try:
                c, x, y, w, h = map(float, v)
                if not (c.is_integer() and 0 <= c < nc and
                        0 <= x <= 1 and 0 <= y <= 1 and
                        0 < w <= 1 and 0 < h <= 1):
                    invalides.append(f"{fichier.name}, ligne {n}")
            except ValueError:
                invalides.append(f"{fichier.name}, ligne {n}")

    print(split.upper(), ":", len(invalides), "lignes invalides")
    if invalides:
        print("  Exemples :", invalides[:10])

In [ ]:
# Nettoyer les lignes YOLO invalides, puis supprimer les labels vides et leurs images
for split in ["train", "valid", "test"]:
    labels_dir = DATASET_DIR / split / "labels"
    images_dir = DATASET_DIR / split / "images"
    if not labels_dir.exists():
        continue

    lignes_supprimees = 0
    fichiers_supprimes = 0

    for label in list(labels_dir.glob("*.txt")):
        lignes_brutes = [l for l in label.read_text(encoding="utf-8").splitlines() if l.strip()]
        lignes_valides = []

        for ligne in lignes_brutes:
            v = ligne.split()
            if len(v) != 5:
                lignes_supprimees += 1
                continue
            try:
                c, x, y, w, h = map(float, v)
                if (c.is_integer() and 0 <= c < nc and
                        0 <= x <= 1 and 0 <= y <= 1 and
                        0 < w <= 1 and 0 < h <= 1):
                    lignes_valides.append(ligne.strip())
                else:
                    lignes_supprimees += 1
            except ValueError:
                lignes_supprimees += 1

        if not lignes_valides:
            nom_image = label.stem
            for ext in [".jpg", ".jpeg", ".png", ".webp"]:
                image = images_dir / (nom_image + ext)
                if image.exists():
                    image.unlink()
                    break
            label.unlink()
            fichiers_supprimes += 1
        elif len(lignes_valides) != len(lignes_brutes):
            label.write_text("\n".join(lignes_valides) + "\n", encoding="utf-8")

    cache_file = DATASET_DIR / split / "labels.cache"
    if cache_file.exists() and (lignes_supprimees > 0 or fichiers_supprimes > 0):
        cache_file.unlink()

    print(f"{split.upper()} : {lignes_supprimees} lignes invalides retirées, {fichiers_supprimes} paires image/label supprimées")

In [ ]:
import yaml
from collections import Counter
import matplotlib.pyplot as plt

with open(DATASET_DIR / "data_1.yaml", "r") as f:
    names = yaml.safe_load(f)["names"]

compteur = Counter()

for fichier in (DATASET_DIR / "train" / "labels").glob("*.txt"):
    for ligne in fichier.read_text().splitlines():
        if ligne.strip():
            compteur[int(ligne.split()[0])] += 1

plt.figure(figsize=(12, 5))
plt.bar(names, [compteur[i] for i in range(len(names))])
plt.title("Distribution des classes - Train")
plt.xlabel("Classes")
plt.ylabel("Nombre d'objets")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

### 3. Création des ensembles d’entraînement, de validation et de test

In [ ]:
#Identifier les donnes apres nettoyage
for split in ["train", "valid"]:
    images = list((DATASET_DIR / split / "images").glob("*"))
    labels = list((DATASET_DIR / split / "labels").glob("*.txt"))

    print(f"{split.upper()} :")
    print(f"Images : {len(images)}")
    print(f"Annotations : {len(labels)}")

In [ ]:
from sklearn.model_selection import train_test_split
import shutil

EXPECTED = {"train": 2297, "valid": 298, "test": 405}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def image_files(split):
    return sorted((p for p in (DATASET_DIR / split / "images").iterdir()
                   if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS), key=lambda p: p.name)

def label_files(split):
    return {p.stem: p for p in (DATASET_DIR / split / "labels").glob("*.txt")}

valid_images = image_files("valid")
valid_labels = label_files("valid")
if len(valid_images) != EXPECTED["valid"] or len(valid_labels) != EXPECTED["valid"]:
    raise ValueError(f"Avant d?coupage, valid doit contenir {EXPECTED['valid']} images/labels; trouv? {len(valid_images)}/{len(valid_labels)}.")

train_images = image_files("train")
test_img_dir = DATASET_DIR / "test" / "images"
test_lbl_dir = DATASET_DIR / "test" / "labels"
test_images = image_files("test") if test_img_dir.exists() else []
source_images = sorted(train_images + test_images, key=lambda p: p.name)
source_labels = {**label_files("train"), **label_files("test")}
if len(source_images) != EXPECTED["train"] + EXPECTED["test"]:
    raise ValueError(f"Le pool train+test doit contenir {EXPECTED['train'] + EXPECTED['test']} images; trouv?: {len(source_images)}.")
if len({p.stem for p in source_images}) != len(source_images):
    raise ValueError("Noms d'images non uniques entre train et test; aucun fichier n'a ?t? d?plac?.")
missing = [p.name for p in source_images if p.stem not in source_labels]
if missing:
    raise ValueError(f"Annotations manquantes avant d?coupage (exemples): {missing[:10]}")

if not test_images:
    if list(test_lbl_dir.glob("*.txt")) if test_lbl_dir.exists() else False:
        raise ValueError("test/labels contient des fichiers alors que test/images est vide; v?rifiez manuellement.")
    if len(train_images) != EXPECTED["train"] + EXPECTED["test"]:
        raise ValueError("Le dossier train ne contient pas le pool source attendu de 2702 images.")
    _, test_images = train_test_split(
        train_images,
        test_size=EXPECTED["test"],
        random_state=42,
    )
    test_img_dir.mkdir(parents=True, exist_ok=True)
    test_lbl_dir.mkdir(parents=True, exist_ok=True)
    train_lbl_dir = DATASET_DIR / "train" / "labels"
    for img in test_images:
        label = source_labels[img.stem]
        shutil.move(str(img), str(test_img_dir / img.name))
        shutil.move(str(label), str(test_lbl_dir / label.name))
else:
    print("Split test existant conserv?; le tirage initial utilise random_state=42.")

for split, expected_count in EXPECTED.items():
    images = image_files(split)
    labels = label_files(split)
    if len(images) != expected_count or len(labels) != expected_count:
        raise AssertionError(f"{split}: attendu {expected_count} images/labels; obtenu {len(images)}/{len(labels)}")
    if {p.stem for p in images} != set(labels):
        raise AssertionError(f"Les images et annotations ne correspondent pas pour {split}.")
    print(f"{split.upper()} : {len(images)} images, {len(labels)} annotations")


In [ ]:
import os
import yaml

# YOLOv9 r?sout `path` depuis la racine du d?p?t YOLO, pas depuis ce fichier YAML.
YOLO_ROOT = (DATASET_DIR.parent / "yolov9").resolve()
if not YOLO_ROOT.is_dir():
    raise FileNotFoundError(f"D?p?t YOLOv9 introuvable : {YOLO_ROOT}")

yaml_path = DATASET_DIR / "data_1.yaml"
with yaml_path.open("r", encoding="utf-8") as f:
    cfg_yaml = yaml.safe_load(f)

cfg_yaml["path"] = os.path.relpath(DATASET_DIR.resolve(), YOLO_ROOT).replace("\\", "/")
cfg_yaml["train"] = "train/images"
cfg_yaml["val"] = "valid/images"
cfg_yaml["test"] = "test/images"

resolved_dataset = (YOLO_ROOT / cfg_yaml["path"]).resolve()
if resolved_dataset != DATASET_DIR.resolve():
    raise AssertionError(f"Chemin dataset incorrect : {resolved_dataset}")
for key, split in [("train", "train"), ("val", "valid"), ("test", "test")]:
    resolved_images = (resolved_dataset / cfg_yaml[key]).resolve()
    if not resolved_images.is_dir():
        raise FileNotFoundError(f"Chemin YAML {key} introuvable : {resolved_images}")
    print(f"{key:5s} -> {resolved_images} ({len(list(resolved_images.iterdir()))} fichiers)")

with yaml_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(cfg_yaml, f, sort_keys=False, allow_unicode=True)
print("Configuration YOLOv9 mise ? jour :", yaml_path)


In [ ]:
# Recherche et correction d?terministe des doublons exacts / images tr?s similaires.
# Les copies concern?es rejoignent train; des images isol?es les remplacent dans valid/test.
import hashlib
import random
import shutil
from collections import Counter
import numpy as np
from PIL import Image

HASH_DISTANCE_LIMIT = 4

def compute_dhash(img_path, hash_size=8):
    with Image.open(img_path) as im:
        im = im.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
        arr = np.asarray(im, dtype=np.int16)
    bits = arr[:, 1:] > arr[:, :-1]
    value = 0
    for bit in bits.flatten():
        value = (value << 1) | int(bit)
    return value

def class_profile(label_path):
    return Counter(int(line.split()[0]) for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip())

def scan_images():
    scanned = {}
    for split in ["train", "valid", "test"]:
        scanned[split] = []
        for path in image_files(split):
            scanned[split].append((path, hashlib.sha256(path.read_bytes()).hexdigest(), compute_dhash(path)))
    return scanned

def find_near_pairs(scanned):
    pairs = []
    for split_a, split_b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
        for path_a, sha_a, hash_a in scanned[split_a]:
            for path_b, sha_b, hash_b in scanned[split_b]:
                distance = (hash_a ^ hash_b).bit_count()
                if sha_a == sha_b or distance <= HASH_DISTANCE_LIMIT:
                    pairs.append((split_a, path_a, split_b, path_b, distance))
    return pairs

records = scan_images()
near_pairs = find_near_pairs(records)
for split_a, split_b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    exact_count = sum(1 for a, pa, b, pb, _ in near_pairs
                      if {a, b} == {split_a, split_b}
                      and hashlib.sha256(pa.read_bytes()).hexdigest() == hashlib.sha256(pb.read_bytes()).hexdigest())
    print(f"Doublons exacts {split_a}/{split_b}: {exact_count}")
print(f"Paires similaires avant correction: {len(near_pairs)}")

if near_pairs:
    marked = {"valid": set(), "test": set()}
    for split_a, path_a, split_b, path_b, _ in near_pairs:
        if split_a != "train":
            marked[split_a].add(path_a)
        if split_b != "train":
            marked[split_b].add(path_b)

    all_records = [(split, path, sha, dhash)
                   for split, rows in records.items()
                   for path, sha, dhash in rows]
    replacement_candidates = []
    marked_train = {path for split_a, path_a, split_b, path_b, _ in near_pairs
                    for split, path in [(split_a, path_a), (split_b, path_b)] if split == "train"}
    for path, _, image_hash in records["train"]:
        if path in marked_train:
            continue
        if all((image_hash ^ other_hash).bit_count() > HASH_DISTANCE_LIMIT
               for _, other_path, _, other_hash in all_records if other_path != path):
            replacement_candidates.append(path)

    required = sum(len(paths) for paths in marked.values())
    if len(replacement_candidates) < required:
        raise RuntimeError(f"Il faut {required} images de remplacement isol?es; seules {len(replacement_candidates)} sont disponibles.")

    rng = random.Random(42)
    rng.shuffle(replacement_candidates)
    label_map = {split: label_files(split) for split in ["train", "valid", "test"]}
    profile_cache = {}
    def cached_profile(path):
        if path not in profile_cache:
            profile_cache[path] = class_profile(path)
        return profile_cache[path]
    replacement_profiles = {path: cached_profile(label_map["train"][path.stem])
                            for path in replacement_candidates}
    replacement_map = {}
    for target_split in ["valid", "test"]:
        for duplicate_path in sorted(marked[target_split], key=lambda path: path.name):
            target_profile = cached_profile(label_map[target_split][duplicate_path.stem])
            best_index = min(
                range(len(replacement_candidates)),
                key=lambda index: (
                    sum(abs(replacement_profiles[replacement_candidates[index]][class_id] - target_profile[class_id])
                        for class_id in set(replacement_profiles[replacement_candidates[index]]) | set(target_profile)),
                    index,
                ),
            )
            replacement_map[(target_split, duplicate_path)] = replacement_candidates.pop(best_index)

    future = {split: {path: dhash for path, _, dhash in rows} for split, rows in records.items()}
    for split, duplicate_path in [(split, path) for split, paths in marked.items() for path in paths]:
        future[split].pop(duplicate_path, None)
        future["train"][duplicate_path] = next(dhash for path, _, dhash in records[split] if path == duplicate_path)
    for (split, _), replacement_path in replacement_map.items():
        future["train"].pop(replacement_path, None)
        replacement_hash = next(dhash for path, _, dhash in records["train"] if path == replacement_path)
        future[split][replacement_path] = replacement_hash
    for split_a, split_b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
        conflicts = [(path_a.name, path_b.name)
                     for path_a, hash_a in future[split_a].items()
                     for path_b, hash_b in future[split_b].items()
                     if (hash_a ^ hash_b).bit_count() <= HASH_DISTANCE_LIMIT]
        if conflicts:
            raise RuntimeError(f"Le remplacement propos? laisse des paires similaires entre {split_a}/{split_b}: {conflicts[:5]}")

    all_paths = [path for rows in records.values() for path, _, _ in rows]
    if len({path.name for path in all_paths}) != len(all_paths):
        raise RuntimeError("Des noms de fichiers se r?p?tent entre ensembles; ?change annul?.")
    if any(path.stem not in label_map[split]
           for split, paths in marked.items() for path in paths):
        raise RuntimeError("Une image candidate n'a pas d'annotation; ?change annul?.")

    temp_dir = DATASET_DIR / ".dedup_swap_tmp"
    if temp_dir.exists():
        raise FileExistsError(f"Dossier temporaire d?j? pr?sent; v?rifiez-le avant de continuer: {temp_dir}")
    temp_images = temp_dir / "images"
    temp_labels = temp_dir / "labels"
    temp_images.mkdir(parents=True)
    temp_labels.mkdir()
    moves = []
    for split, paths in marked.items():
        for path in paths:
            moves.append((path, DATASET_DIR / "train" / "images" / path.name,
                          label_map[split][path.stem], DATASET_DIR / "train" / "labels" / label_map[split][path.stem].name))
    for (split, _), path in replacement_map.items():
        moves.append((path, DATASET_DIR / split / "images" / path.name,
                      label_map["train"][path.stem], DATASET_DIR / split / "labels" / label_map["train"][path.stem].name))

    staged = []
    installed = []
    try:
        for index, (source_image, destination_image, source_label, destination_label) in enumerate(moves):
            temp_image = temp_images / f"{index:04d}_{source_image.name}"
            temp_label = temp_labels / f"{index:04d}_{source_label.name}"
            shutil.move(str(source_image), str(temp_image))
            staged.append((source_image, temp_image))
            shutil.move(str(source_label), str(temp_label))
            staged.append((source_label, temp_label))
            installed.append((temp_image, destination_image))
            installed.append((temp_label, destination_label))
        for temporary, destination in installed:
            shutil.move(str(temporary), str(destination))
    except Exception:
        for temporary, destination in reversed(installed):
            if destination.exists():
                shutil.move(str(destination), str(temporary))
        for original, temporary in reversed(staged):
            if temporary.exists():
                original.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(temporary), str(original))
        raise
    finally:
        if temp_images.exists() and not any(temp_images.iterdir()):
            temp_images.rmdir()
        if temp_labels.exists() and not any(temp_labels.iterdir()):
            temp_labels.rmdir()
        if temp_dir.exists() and not any(temp_dir.iterdir()):
            temp_dir.rmdir()

    print(f"Remplacements appliqu?s : {len(marked['valid'])} dans valid, {len(marked['test'])} dans test.")
    records = scan_images()
    near_pairs = find_near_pairs(records)
    print(f"Paires similaires apr?s correction: {len(near_pairs)}")
    if near_pairs:
        raise AssertionError("Des images similaires restent r?parties entre plusieurs ensembles.")
else:
    print("Aucune paire similaire ? corriger.")


### 4. Vérification finale après nettoyage et analyse des déséquilibres de classes

In [ ]:
# Contr?le final strict : effectifs, correspondance image/label et int?grit? des annotations.
EXPECTED = {"train": 2297, "valid": 298, "test": 405}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
from PIL import Image
errors = []

for split, expected_count in EXPECTED.items():
    img_dir = DATASET_DIR / split / "images"
    lbl_dir = DATASET_DIR / split / "labels"
    images = {p.stem for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS}
    labels = {p.stem for p in lbl_dir.glob("*.txt")}
    if len(images) != expected_count or len(labels) != expected_count:
        errors.append(f"{split}: effectifs attendus {expected_count}/{expected_count}, trouv?s {len(images)}/{len(labels)}")

    missing_labels = images - labels
    orphan_labels = labels - images
    empty_labels = 0
    invalid_rows = 0
    corrupt_images = []
    for image_path in img_dir.iterdir():
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            try:
                with Image.open(image_path) as image:
                    image.verify()
            except (OSError, ValueError):
                corrupt_images.append(image_path.name)
    for label_file in lbl_dir.glob("*.txt"):
        lines = [line for line in label_file.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not lines:
            empty_labels += 1
        for line in lines:
            values = line.split()
            try:
                if len(values) != 5:
                    raise ValueError
                class_id = float(values[0])
                x, y, width, height = map(float, values[1:])
                if not class_id.is_integer() or not (0 <= int(class_id) < nc):
                    raise ValueError
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < width <= 1 and 0 < height <= 1):
                    raise ValueError
            except ValueError:
                invalid_rows += 1

    if missing_labels or orphan_labels or empty_labels or invalid_rows or corrupt_images:
        errors.append(f"{split}: sans label={len(missing_labels)}, sans image={len(orphan_labels)}, vides={empty_labels}, lignes invalides={invalid_rows}, images corrompues={len(corrupt_images)}")
    print(f"{split.upper()} : images={len(images)}, labels={len(labels)}, sans label={len(missing_labels)}, sans image={len(orphan_labels)}, vides={empty_labels}, lignes invalides={invalid_rows}, images corrompues={len(corrupt_images)}")

if near_pairs:
    errors.append(f"{len(near_pairs)} paires d'images similaires restent ? examiner")
if errors:
    raise AssertionError("Validation dataset ?chou?e :\n- " + "\n- ".join(errors))
print("Effectifs, annotations et s?paration des images : VALID?S")


In [ ]:
# Analyse détaillée de la répartition des classes et des déséquilibres (Train / Valid / Test)
import pandas as pd

repartition = {split: Counter() for split in ["train", "valid", "test"]}
for split in ["train", "valid", "test"]:
    for f in (DATASET_DIR / split / "labels").glob("*.txt"):
        for ligne in f.read_text(encoding="utf-8").splitlines():
            if ligne.strip():
                repartition[split][int(ligne.split()[0])] += 1

df_classes = pd.DataFrame({
    "class_id": list(range(nc)),
    "classe": names,
    "train": [repartition["train"][i] for i in range(nc)],
    "valid": [repartition["valid"][i] for i in range(nc)],
    "test": [repartition["test"][i] for i in range(nc)],
})
df_classes["total"] = df_classes["train"] + df_classes["valid"] + df_classes["test"]
df_classes["pct_total"] = (df_classes["total"] / df_classes["total"].sum() * 100).round(2)
df_classes = df_classes.sort_values("total", ascending=False).reset_index(drop=True)

print(df_classes.to_string(index=False))
print("\n--- Diagnostic du déséquilibre ---")
print(f"Classe majoritaire : {df_classes.iloc[0]['classe']} ({df_classes.iloc[0]['total']} objets)")
print(f"Classe minoritaire : {df_classes.iloc[-1]['classe']} ({df_classes.iloc[-1]['total']} objets)")
print(f"Ratio max/min global : {df_classes['total'].max() / df_classes['total'].min():.1f}x")
print("Classes rares (< 100 objets au total) :", df_classes[df_classes["total"] < 100]["classe"].tolist())
print("Classes absentes de valid :", df_classes[df_classes["valid"] == 0]["classe"].tolist())
print("Classes absentes de test  :", df_classes[df_classes["test"] == 0]["classe"].tolist())

ax = df_classes.set_index("classe")[["train", "valid", "test"]].plot(kind="bar", stacked=True, figsize=(13, 5))
ax.set_title("Répartition des 21 classes par split (après nettoyage)")
ax.set_ylabel("Nombre de boîtes englobantes")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()